# Biot consolidation of a unit square

This notebook solves a two-dimensional, transiently coupled Biot consolidation problem on the domain

$$
\Omega = (0,L) \times (0,H), \qquad L=H=1.
$$

The unknown fields are the displacement

$$
\boldsymbol u : \Omega \times [0,T] \to \mathbb{R}^2
$$

and the pore pressure

$$
p : \Omega \times [0,T] \to \mathbb{R}.
$$


In [ ]:
import jax
from jax import config
import jax.numpy as jnp

from autopdex import SimState, dae, spaces

config.update("jax_enable_x64", True)

In [2]:
L, H = 1.0, 1.0
vertices = [[0.0, 0.0], [L, 0.0], [L, H], [0.0, H]]
n_elements = (40, 40)

E, nu = 10.0e6, 0.30
lam = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
mu = E / (2.0 * (1.0 + nu))
alpha, c0 = 1.0, 1.0e-8
K = 1.0e-10 * jnp.eye(2)

t_final, num_time_steps = 1e3, 50
dt = t_final / num_time_steps
I = jnp.eye(2)

## Weak form

For small strains,

$$
\boldsymbol\varepsilon(\boldsymbol u)
= \frac{1}{2}\left(\nabla \boldsymbol u + \nabla \boldsymbol u^T\right).
$$

The elastic strain energy density is

$$
\psi(\boldsymbol\varepsilon)
= \frac{1}{2}\lambda\,\mathrm{tr}(\boldsymbol\varepsilon)^2
+ \mu\,\boldsymbol\varepsilon:\boldsymbol\varepsilon.
$$

This yields the effective stress

$$
\boldsymbol\sigma' = \frac{\partial \psi}{\partial \boldsymbol\varepsilon},
$$

and the total stress is defined as

$$
\boldsymbol\sigma = \boldsymbol\sigma' - \alpha p \boldsymbol I
$$

The Darcy flux is

$$
\boldsymbol q = -\boldsymbol K \nabla p.
$$

The implemented weak form is: find $(\boldsymbol u,p)$ such that, for all test functions $(\boldsymbol v,w)$,

$$
\int_\Omega \boldsymbol\sigma : \boldsymbol\varepsilon(\boldsymbol v)\,\mathrm d\Omega
- \int_{\Gamma} \bar{\boldsymbol t}\cdot \boldsymbol v\,\mathrm d\Gamma
= 0,
$$

and

$$
\int_\Omega
\left(c_0\dot p + \alpha \frac{\partial}{\partial t}\nabla\cdot\boldsymbol u\right) w
+ \nabla w \cdot \boldsymbol K \nabla p
\,\mathrm d\Omega
= 0.
$$

In the code, the volume contribution of the weak form is defined in `weak_form`; the Neumann load is applied separately with `add_weak_bc`.


In [3]:
def strain_energy_density(eps):
    return 0.5 * lam * jnp.trace(eps) ** 2 + mu * jnp.sum(eps ** 2)

def weak_form(ctx: SimState.ModelContext):
    u_fun, p_fun = ctx.trial_ansatz["displacement"], ctx.trial_ansatz["pressure"]
    x, t = ctx.x_int, ctx.t

    grad_u_fun = jax.jacfwd(u_fun)
    grad_u = grad_u_fun(x, t)
    eps_u = 0.5 * (grad_u + grad_u.T)
    p = p_fun(x, t)
    grad_p, p_dot = jax.jacrev(p_fun, (0, 1))(x, t)

    sigma = jax.jacrev(strain_energy_density)(eps_u) - alpha * p * I

    if ctx.mode == "output":
        return {
            "strain": eps_u,
            "stress": sigma,
            "darcy_flux": -K @ grad_p,
        }

    v_fun, w_fun = ctx.test_ansatz["displacement"], ctx.test_ansatz["pressure"]
    grad_v, grad_w = jax.jacfwd(v_fun)(x), jax.jacfwd(w_fun)(x)
    eps_v = 0.5 * (grad_v + grad_v.T)
    w = w_fun(x)
    div_u_dot = jax.jacfwd(lambda tau: jnp.trace(grad_u_fun(x, tau)))(t)

    return jnp.sum(sigma * eps_v) + (c0 * p_dot + alpha * div_u_dot) * w + grad_w @ K @ grad_p

## Boundary conditions


<table>
<tr>
<td style="width: 50%; vertical-align: middle;">

For the displacement, the strong boundary conditions are

$$
u_x = 0
\quad \text{on} \quad \{x=0\}\cup\{x=L\},
$$

$$
u_y = 0
\quad \text{on} \quad \{y=0\}.
$$

For the pore pressure, the drained boundary condition is prescribed on the top edge:

$$
p = 0
\quad \text{on} \quad \{y=H\}.
$$

In addition, the constant surface load

$$
\bar{\boldsymbol t} = \begin{bmatrix}0\\-100\,\mathrm{kPa}\end{bmatrix}
\quad \text{on} \quad \{y=H\}
$$

is applied on the top edge. On the remaining pressure boundaries, the natural no-flow boundary condition follows:

$$
\boldsymbol q\cdot\boldsymbol n = 0.
$$


</td>
<td style="width: 50%; vertical-align: middle;">

<img src="./visualizations/biot_consolidation.svg" style="width: 100%;">

</td>
</tr>
</table>





In [4]:
sim = SimState({
    "displacement": spaces.H1(order=2, dim=2, field_dimension=2),
    "pressure": spaces.H1(order=1, dim=2, field_dimension=1),
})

sim.add_structured_mesh(n_elements, vertices, "quad")
sim.add_temporal_discretization({
    "displacement": dae.BackwardEuler(),
    "pressure": dae.BackwardEuler(),
})
sim.add_model("__all__", "weak form", weak_form)

sim.add_strong_bc(
    "displacement",
    on_boundary_fun=lambda x: jnp.logical_or(jnp.isclose(x[0], 0.0), jnp.isclose(x[0], L)),
    value_fun=lambda x, t, s: 0.,
    index=0,
)
sim.add_strong_bc(
    "displacement",
    on_boundary_fun=lambda x: jnp.isclose(x[1], 0.0),
    value_fun=lambda x, t, s: 0.,
    index=1,
)
sim.add_strong_bc(
    "pressure",
    on_boundary_fun=lambda x: jnp.isclose(x[1], H),
    value_fun=lambda x, t, s: jnp.zeros((1,)),
)
sim.add_weak_bc(
    "displacement",
    on_boundary_fun=lambda x: jnp.isclose(x[1], H),
    value_fun=lambda x, t, s: jnp.asarray([0.0, -100.0e3]),
)

In [ ]:
sim.set_postprocessing_policy(
    dae.SaveEquidistantPolicy(10),
    result_folder_name="biot_consolidation.res",
)
sim.initialize(1)
sim.prepare()

In [ ]:
result = sim.run(dt0=dt, time_span=t_final, num_time_steps=num_time_steps)

Linear solver: Pardiso(lu).
Iteration 1, Residual norm: 3.2790375156968575e-12
 
Iteration 1, Residual norm: 1.9836404764119437e-12
Progress: 4%, Time: 4.00e+01, dt: 2.00e+01, iterations: 1
 
Iteration 1, Residual norm: 2.2959053785144467e-12
 
Iteration 1, Residual norm: 2.5441995544474102e-12
 
Iteration 1, Residual norm: 2.829674488451506e-12
Progress: 10%, Time: 1.00e+02, dt: 2.00e+01, iterations: 1
 
Iteration 1, Residual norm: 2.9656626858298675e-12
 
Iteration 1, Residual norm: 3.176456184379287e-12
 
Iteration 1, Residual norm: 3.492888465056613e-12
Progress: 16%, Time: 1.60e+02, dt: 2.00e+01, iterations: 1
 
Iteration 1, Residual norm: 3.670292449868166e-12
 
Iteration 1, Residual norm: 4.075062920096286e-12
 
Iteration 1, Residual norm: 4.115568466999836e-12
Progress: 22%, Time: 2.20e+02, dt: 2.00e+01, iterations: 1
 
Iteration 1, Residual norm: 4.589469059177754e-12
 
Iteration 1, Residual norm: 4.644667220129573e-12
 
Iteration 1, Residual norm: 4.9968218070280015e-12
Progr